In [ ]:
install uv:
# create a new virtual environment
uv venv -p 3.11
source .venv/bin/activate

# install specforge
uv pip install -v -e. --prerelease=allow

In [ ]:
parquet to jsonl：
python ./scripts/parquet2jsonl.py \
--input-dir /inspire/hdd/project/inference-chip/xujiaming-253308120313/whz/datasets/train/nvidia/Nemotron-Post-Training-Dataset-v2/data \
--output-dir /inspire/hdd/project/inference-chip/xujiaming-253308120313/whz/spec-forge/cache/dataset/nemotron/origin_jsonl \
--num-samples 600000 \
--input-dir /share/wanghanzhen/.cache/huggingface/hub/datasets--nvidia--Nemotron-Post-Training-Dataset-v2/snapshots/5c89e01dd720ae0f4058445ed49c5fb68a03c76e/data \
--output-dir /share/wanghanzhen/SpeculativeDecoding/NIPS26/SpecForge/cache/dataset \
--num-samples 10 \
--seed 42

In [ ]:
launch server:
CUDA_VISIBLE_DEVICES=0 python3 -m sglang.launch_server \
    --model /inspire/hdd/project/inference-chip/xujiaming-253308120313/whz/models/Qwen/Qwen3-8B \
    --cuda-graph-bs 1 2 4 8 16 32 64 128 \
    --dtype bfloat16 \
    --mem-frac=0.8 \
    --port 30001

CUDA_VISIBLE_DEVICES=1 python3 -m sglang.launch_server \
    --model /inspire/hdd/project/inference-chip/xujiaming-253308120313/whz/models/Qwen/Qwen3-8B \
    --cuda-graph-bs 1 2 4 8 16 32 64 128 \
    --dtype bfloat16 \
    --mem-frac=0.8 \
    --port 30002

CUDA_VISIBLE_DEVICES=2 python3 -m sglang.launch_server \
    --model /inspire/hdd/project/inference-chip/xujiaming-253308120313/whz/models/Qwen/Qwen3-8B \
    --cuda-graph-bs 1 2 4 8 16 32 64 128 \
    --dtype bfloat16 \
    --mem-frac=0.8 \
    --port 30003

CUDA_VISIBLE_DEVICES=3 python3 -m sglang.launch_server \
    --model /inspire/hdd/project/inference-chip/xujiaming-253308120313/whz/models/Qwen/Qwen3-8B \
CUDA_VISIBLE_DEVICES=1 python3 -m sglang.launch_server \
    --model /share/public/public_models/Qwen3-8B \
    --cuda-graph-bs 1 2 4 8 16 32 64 128 \
    --dtype bfloat16 \
    --mem-frac=0.8 \
    --port 30004


regenerate dataset:

In [ ]:
python scripts/regenerate_train_data.py \
    --model /share/public/public_models/Qwen3-8B \
    --concurrency 64 \
    --server-address localhost:30004 localhost:30001 localhost:30002 localhost:30003 \
    --temperature 0.0 \
    --input-file-path ./cache/dataset/nemotron/origin_jsonl/Nemotron_600000.jsonl 


python scripts/regenerate_train_data.py \
    --model /share/public/public_models/Qwen3-8B \
    --concurrency 64 \
    --max-tokens 10240 \
    --server-address localhost:30006 \
    --temperature 0.0 \
    --input-file-path ./cache/dataset/nemotron_preview.jsonl \
    --output-file-path ./cache/dataset/nemotron_test_train_regen.jsonl

claude

In [ ]:
bash /share/wanghanzhen/claude/run_claude.sh 